In [15]:
rec_limit = 1000 + SEED
target_class = (SEED % 4) + 2
filtered_records = []
status_count = {}
url_counts = {}

with open(DATA_DIR / "weblog.csv", "r", encoding="utf-8") as f:
    f.readline()
    for idx, line in enumerate(f):
        if idx >= rec_limit:
            break
        parts = line.strip().split(",")
        if len(parts) < 4:
            continue
        ip, tim, url, st = parts[0], parts[1], parts[2], parts[3]
        if st.startswith(str(target_class)):
            row = {"IP": ip, "Time": tim, "URL": url, "Status": st}
            filtered_records.append(row)
            status_count[st] = status_count.get(st, 0) + 1
            url_counts[url] = url_counts.get(url, 0) + 1

top_urls = sorted(url_counts.items(), key=lambda x: x[1], reverse=True)[:5]

csv_path = OUTPUT_DIR / f"{LAST_NAME}_weblog_filtered.csv"
with open(csv_path, "w", encoding="utf-8") as fw:
    fw.write("IP,Time,URL,Status\n")
    for r in filtered_records:
        fw.write(f"{r['IP']},{r['Time']},{r['URL']},{r['Status']}\n")

summary = {
    "Records_processed": rec_limit,
    "Target_status_class": f"{target_class}xx",
    "Records_in_target_class": len(filtered_records),
    "Count_per_status_code": status_count,
    "Top5_URLs": dict(top_urls)
}
json_path = OUTPUT_DIR / f"{LAST_NAME}_weblog_summary.json"
with open(json_path, "w", encoding="utf-8") as fj:
    json.dump(summary, fj, indent=2)

shutil.copy2(json_path, BACKUP_DIR / json_path.name)
print(summary)
print("Backup exists:", (BACKUP_DIR / json_path.name).exists())


{'Records_processed': 1009, 'Target_status_class': '3xx', 'Records_in_target_class': 164, 'Count_per_status_code': {'302': 143, '304': 21}, 'Top5_URLs': {'GET / HTTP/1.1': 45, 'POST /process.php HTTP/1.1': 35, 'GET /contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest HTTP/1.1': 34, 'GET /home.php HTTP/1.1': 9, 'GET /logout.php HTTP/1.1': 7}}
Backup exists: True


In [16]:
selected = []
sum_batt = 0.0
sum_data = 0.0
count = 0

target_os = "iOS" if SEED % 2 == 1 else "Android"
target_behavior = (SEED % 5) + 1

with open(DATA_DIR / "user_behavior_dataset.csv", "r", encoding="utf-8") as f:
    f.readline()
    for line in f:
        parts = line.strip().split(",")
        if len(parts) < 6:
            continue
        uid, model, osys, batt, datau, bclass = parts[0], parts[1], parts[2], parts[3], parts[4], parts[5]
        try:
            if osys == target_os and int(bclass) == target_behavior:
                sum_batt += float(batt)
                sum_data += float(datau)
                count += 1
                selected.append({
                    "User_ID": uid,
                    "Device_Model": model,
                    "OS": osys,
                    "Battery_Drain": batt,
                    "Data_Usage": datau
                })
        except ValueError:
            continue

avg_batt = sum_batt / count if count > 0 else 0
avg_data = sum_data / count if count > 0 else 0

csv_path = OUTPUT_DIR / f"{LAST_NAME}_mobile_filtered.csv"
with open(csv_path, "w", encoding="utf-8") as fw:
    fw.write("User_ID,Device_Model,Operating_System,Battery_Drain(mAh/day),Data_Usage(MB/day)\n")
    for r in selected:
        fw.write(f"{r['User_ID']},{r['Device_Model']},{r['OS']},{r['Battery_Drain']},{r['Data_Usage']}\n")

summary = {
    "OS_and_behavior_class": f"{target_os}, Class {target_behavior}",
    "Matching_records": count,
    "Avg_battery_drain": round(avg_batt, 2),
    "Avg_data_usage": round(avg_data, 2),
    "Count_verification": count == len(selected)
}
json_path = OUTPUT_DIR / f"{LAST_NAME}_mobile_summary.json"
with open(json_path, "w", encoding="utf-8") as fj:
    json.dump(summary, fj, indent=2)

shutil.copy2(csv_path, BACKUP_DIR / csv_path.name)
print(summary)
print("Backup size:", (BACKUP_DIR / csv_path.name).stat().st_size if count > 0 else 0)


{'OS_and_behavior_class': 'iOS, Class 5', 'Matching_records': 0, 'Avg_battery_drain': 0, 'Avg_data_usage': 0, 'Count_verification': True}
Backup size: 0


In [17]:
incidents = []
type_counts = {}
total_loss = 0.0
target_year = 2015 + SEED

with open(DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv", "r", encoding="utf-8") as f:
    f.readline()
    for line in f:
        parts = line.strip().split(",")
        if len(parts) < 4:
            continue
        try:
            yr = int(parts[0])
            loss = float(parts[2])
            atk_type = parts[3]
        except (ValueError, IndexError):
            continue
        if yr == target_year:
            incidents.append({"year": yr, "attack_type": atk_type, "loss_mil": loss})
            type_counts[atk_type] = type_counts.get(atk_type, 0) + 1
            total_loss += loss

avg_loss = total_loss / len(incidents) if len(incidents) > 0 else 0
top5 = sorted(incidents, key=lambda x: -x["loss_mil"])[:5]

csv_path = OUTPUT_DIR / f"{LAST_NAME}_cyber_filtered.csv"
with open(csv_path, "w", encoding="utf-8") as fw:
    fw.write("Year,Attack_Type,Loss_Million_USD\n")
    for r in incidents:
        fw.write(f"{r['year']},{r['attack_type']},{r['loss_mil']}\n")

summary = {
    "Target_year": target_year,
    "Incidents_in_target_year": len(incidents),
    "Count_per_attack_type": type_counts,
    "Total_loss_milUSD": round(total_loss, 2),
    "Avg_loss_milUSD": round(avg_loss, 2),
    "Top5_costliest": [f"{i['year']} – {i['attack_type']} – ${i['loss_mil']:.2f}M" for i in top5]
}
json_path = OUTPUT_DIR / f"{LAST_NAME}_cyber_summary.json"
with open(json_path, "w", encoding="utf-8") as fj:
    json.dump(summary, fj, indent=2)

report_path = OUTPUT_DIR / f"{LAST_NAME}_cyber_report.txt"
with open(report_path, "w", encoding="utf-8") as fr:
    fr.write(f"Target Year: {target_year}\n")
    fr.write(f"Total Loss: ${total_loss:.2f} million\n")
    fr.write(f"Average Loss: ${avg_loss:.2f} million\n")

shutil.copy2(json_path, BACKUP_DIR / json_path.name)
print(summary)
print("Backup exists:", (BACKUP_DIR / json_path.name).exists())


{'Target_year': 2024, 'Incidents_in_target_year': 0, 'Count_per_attack_type': {}, 'Total_loss_milUSD': 0.0, 'Avg_loss_milUSD': 0, 'Top5_costliest': []}
Backup exists: True
